## Importações

In [1]:
import pandas as pd

## Carregamento dos arquivos
- Pesquisa Codificada;
- Dicionário de dados.

In [2]:
df_codificada = pd.read_csv("pesquisa_codificada.csv", sep=";")
df_codificada.head()

,gender,age,height,weight,family_history,favc,fcvc,ncp,caec,smoke,ch2o,scc,faf,tue,calc,mtrans,obesity,imc,ds_imc_oms,ds_imc_dicionario
0,1,21,1.62,64.0,1,2,2,3,2,2,2,2,1,2,1,4,2,24.386526,Peso normal,Peso normal
1,1,21,1.52,56.0,1,2,3,3,2,1,3,1,4,1,2,4,2,24.238227,Peso normal,Peso normal
2,2,23,1.80,77.0,1,2,2,3,2,2,2,2,3,2,3,4,2,23.765432,Peso normal,Peso normal
3,2,27,1.80,87.0,2,2,3,3,2,2,2,2,3,1,3,5,3,26.851852,Sobrepeso,Sobrepeso nível I
4,2,22,1.78,89.8,2,2,2,1,2,2,2,2,1,1,2,4,4,28.342381,Sobrepeso,Sobrepeso nível II


In [3]:
df_dicionario = pd.read_csv("dicionario.csv", sep=";")
df_dicionario.head()

,cd_variavel,ds_variavel,nr_categoria,ds_categoria,sk_categoria
0,gender,Sexo biológico,Female,Feminino,1
1,gender,Sexo biológico,Male,Masculino,2
2,family_history,Histórico familiar de excesso de peso,yes,Há histórico,1
3,family_history,Histórico familiar de excesso de peso,no,Não há histórico,2
4,favc,Consumo frequente de alimentos altamente calór...,yes,Sim,1


## Ajustes iniciais e Seleção de Variáveis (Feature Selection)

A partir de insights obtidos no storytelling, alguns ajustes se fazem necessários como:

### Remoção da coluna Smoke
A coluna smoke possui correlação quase nula (-0,0034) e é extremamente desbalanceada (apenas 44 fumantes), o que não agregará valor preditivo real.

In [4]:
print('DICIONÁRIO DE DADOS')
display(df_dicionario.loc[df_dicionario.cd_variavel == 'smoke'])

print('PESQUISA CODIFICADA','\n', df_codificada.smoke.value_counts(sort=False).sort_index())

DICIONÁRIO DE DADOS


,cd_variavel,ds_variavel,nr_categoria,ds_categoria,sk_categoria
36,smoke,Hábito de fumar,yes,Fuma,1
37,smoke,Hábito de fumar,no,Não fuma,2


PESQUISA CODIFICADA 
 smoke
1      44
2    2067
Name: count, dtype: int64


In [5]:
# removendo a coluna smoke da pesquisa codificada
df_codificada = df_codificada.drop(columns=["smoke"])

# removendo a coluna smoke do dicionario
df_dicionario = df_dicionario.loc[df_dicionario.cd_variavel != 'smoke']

### Tratar o outlier de calc (consumo de bebidas alcoolicas)

A categoria "Sempre" (Always) tem apenas 1 registro, sendo válido agrupar esse registro na categoria "Frequentemente" para evitar que o modelo aprenda uma regra baseada em um único indivíduo.

In [6]:
print('DICIONÁRIO DE DADOS')
display(df_dicionario.loc[df_dicionario.cd_variavel == 'calc'])

print('PESQUISA CODIFICADA','\n', df_codificada.calc.value_counts(sort=False).sort_index())

DICIONÁRIO DE DADOS


,cd_variavel,ds_variavel,nr_categoria,ds_categoria,sk_categoria
32,calc,Consumo de bebida alcoólica,no,Não bebe,1
33,calc,Consumo de bebida alcoólica,Sometimes,Às vezes,2
34,calc,Consumo de bebida alcoólica,Frequently,Frequentemente,3
35,calc,Consumo de bebida alcoólica,Always,Sempre,4


PESQUISA CODIFICADA 
 calc
1     639
2    1401
3      70
4       1
Name: count, dtype: int64


In [7]:
# agrupando a categoria "Sempre" com "Frequentemente" da coluna calc
df_codificada['calc'] = df_codificada['calc'].replace(4,3)

In [8]:
print('PESQUISA CODIFICADA','\n', df_codificada.calc.value_counts(sort=False).sort_index())

PESQUISA CODIFICADA 
 calc
1     639
2    1401
3      71
Name: count, dtype: int64


### Lidando com a Multicolinearidade (IMC vs. Peso)

A correlação entre imc e weight é de 0.93. Quando duas variáveis explicam quase a mesma coisa, mantê-las pode confundir o modelo ou causar overfitting.

Justificativa Clínica: O IMC já é uma medida normalizada (peso dividido pela altura ao quadrado), o que o torna mais relevante para um médico do que o peso isolado.

In [9]:
# removendo as colunas de peso e descrições de IMCs
df_codificada = df_codificada.drop(columns=['weight','ds_imc_oms','ds_imc_dicionario'])

### Agrupamento de Transporte 

Os dados mostram que quem usa transporte ativo tem apenas 5,4% de taxa de obesidade, contra mais de 45% nos meios passivos. Criaremos uma variável binária Transporte Ativo (Caminhada/Bicicleta) e Transporte Passivo (Carro/Moto/Público). 

In [10]:
print('DICIONÁRIO DE DADOS')
display(df_dicionario.loc[df_dicionario.cd_variavel == 'mtrans'])

print('PESQUISA CODIFICADA','\n', df_codificada.mtrans.value_counts(sort=False).sort_index())

DICIONÁRIO DE DADOS


,cd_variavel,ds_variavel,nr_categoria,ds_categoria,sk_categoria
24,mtrans,Meio de transporte habitual,Automobile,Carro,1
25,mtrans,Meio de transporte habitual,Motorbike,Moto,2
26,mtrans,Meio de transporte habitual,Bike,Bicicleta,3
27,mtrans,Meio de transporte habitual,Public_Transportation,Transporte público,4
28,mtrans,Meio de transporte habitual,Walking,A pé,5


PESQUISA CODIFICADA 
 mtrans
1     457
2      11
3       7
4    1580
5      56
Name: count, dtype: int64


In [11]:
# Criar a variável de classificação binária para transportes ativos (mtrans) 
# atribuindo 1 para Bike e Walking, e 0 para os demais meios de transporte

# agrupa os valores segundo a regra e renomeia a coluna
df_codificada.mtrans = df_codificada.mtrans.apply(lambda x : 1 if x in [3,5] else 0 )
df_codificada.rename(columns={'mtrans':'mtrans_ativo'}, inplace=True)

# remove o conteúdo de mtrans anterior do dicionario
df_dicionario = df_dicionario.loc[df_dicionario.cd_variavel != 'mtrans']

# adiciona o novo formato
df_dicionario = pd.concat([df_dicionario, pd.DataFrame({
        "cd_variavel": ["mtrans_ativo", "mtrans_ativo"],
        "ds_variavel": [
            "Meio de transporte habitual",
            "Meio de transporte habitual"
        ],
        "nr_categoria": ["Active", "Passive"],
        "ds_categoria": ["Ativo", "Passivo"],
        "sk_categoria": [1, 0]
    })])


In [12]:
print('DICIONÁRIO DE DADOS')
display(df_dicionario.loc[df_dicionario.cd_variavel == 'mtrans_ativo'])

print('PESQUISA CODIFICADA','\n', df_codificada.mtrans_ativo.value_counts(sort=False).sort_index())

DICIONÁRIO DE DADOS


,cd_variavel,ds_variavel,nr_categoria,ds_categoria,sk_categoria
0,mtrans_ativo,Meio de transporte habitual,Active,Ativo,1
1,mtrans_ativo,Meio de transporte habitual,Passive,Passivo,0


PESQUISA CODIFICADA 
 mtrans_ativo
0    2048
1      63
Name: count, dtype: int64


# Normalização

## StandardScaler

In [13]:
# importa StandardScaler
from sklearn.preprocessing import StandardScaler

In [14]:
# aplica StandardScaler em age height e weight
scaler = StandardScaler()
df_nomalizado = df_codificada.copy()
df_nomalizado[["age", "height", "imc"]] = scaler.fit_transform(df_nomalizado[["age", "height", "imc"]])
df_nomalizado.head()

,gender,age,height,family_history,favc,fcvc,ncp,caec,ch2o,scc,faf,tue,calc,mtrans_ativo,obesity,imc
0,1,-0.471293,-0.874380,1,2,2,3,2,2,2,1,2,1,0,2,-0.663422
1,1,-0.471293,-1.945660,1,2,3,3,2,3,1,4,1,2,0,2,-0.681926
2,2,-0.154194,1.053924,1,2,2,3,2,2,2,3,2,3,0,2,-0.740922
3,2,0.480005,1.053924,2,2,3,3,2,2,2,3,1,3,1,3,-0.355799
4,2,-0.312743,0.839668,2,2,2,1,2,2,2,1,1,2,0,4,-0.169811


## MinMaxScaler

In [15]:
# importando o MinMaxScaler
from sklearn.preprocessing import MinMaxScaler 

In [16]:
#aplicano MinMaxScaler em age height e weight
scaler = MinMaxScaler()
df_nomalizado_2 = df_codificada.copy()
df_nomalizado_2[["age", "height", "imc"]] = scaler.fit_transform(df_nomalizado_2[["age", "height", "imc"]])
df_nomalizado_2.head()

,gender,age,height,family_history,favc,fcvc,ncp,caec,ch2o,scc,faf,tue,calc,mtrans_ativo,obesity,imc
0,1,0.148936,0.320755,1,2,2,3,2,2,2,1,2,1,0,2,0.301089
1,1,0.148936,0.132075,1,2,3,3,2,3,1,4,1,2,0,2,0.297168
2,2,0.191489,0.660377,1,2,2,3,2,2,2,3,2,3,0,2,0.284666
3,2,0.276596,0.660377,2,2,3,3,2,2,2,3,1,3,1,3,0.366278
4,2,0.170213,0.622642,2,2,2,1,2,2,2,1,1,2,0,4,0.405690


Como o MinMaxScaler mantem os valores positivos, acredito que seja mais apropriado, então seguiremos usando o df_nomalizado_2

# Modelagem

## Variáveis de teste

In [17]:
# dividindo os dados em treino e teste
from sklearn.model_selection import train_test_split

In [18]:

X = df_nomalizado_2.drop("obesity", axis=1)
y = df_nomalizado_2["obesity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Métricas de Validação

In [19]:
# importando as metricas de avaliação do modelo
from sklearn.metrics import classification_report, confusion_matrix

## RandomForestClassifier

In [20]:
# importando o RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

In [21]:
# aplicando RandomForestClassifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [22]:
# testando o modelo no conjunto de teste
y_pred = model.predict(X_test)

In [23]:
# exibindo as métricas de avaliação
print("=== CONJUNTO DE TESTE ===")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

=== CONJUNTO DE TESTE ===
Classification Report:
              precision    recall  f1-score   support

           1       0.98      0.96      0.97        56
           2       0.91      0.98      0.95        62
           3       0.98      0.91      0.94        56
           4       0.98      0.98      0.98        50
           5       0.99      1.00      0.99        78
           6       1.00      0.98      0.99        58
           7       1.00      1.00      1.00        63

    accuracy                           0.98       423
   macro avg       0.98      0.97      0.98       423
weighted avg       0.98      0.98      0.98       423


Confusion Matrix:
[[54  2  0  0  0  0  0]
 [ 1 61  0  0  0  0  0]
 [ 0  4 51  1  0  0  0]
 [ 0  0  1 49  0  0  0]
 [ 0  0  0  0 78  0  0]
 [ 0  0  0  0  1 57  0]
 [ 0  0  0  0  0  0 63]]


## XGBoost

In [24]:
# instalando o XGBoost
#!pip install xgboost

In [25]:
# importando o XGBoost
from xgboost import XGBClassifier

In [26]:
# aplicando o xgboost no mesmo conjunto de treino e teste
model_xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric="mlogloss")
y_train = y_train - 1  # muda de [1,2,3,4,5,6,7] para [0,1,2,3,4,5,6]
y_test = y_test - 1    # a mesma explicacao de cima para o conjunto de teste

model_xgb.fit(X_train, y_train)

g:\Meu Drive\pos tech FIAP\fase 4\tech challenge 4\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [12:33:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [27]:
# testando o modelo no conjunto de teste
y_pred_xgb = model_xgb.predict(X_test)

In [28]:
# exibindo as métricas de avaliação
print("=== CONJUNTO DE TESTE - XGBoost ===")
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb)) 

=== CONJUNTO DE TESTE - XGBoost ===
              precision    recall  f1-score   support

           0       1.00      0.93      0.96        56
           1       0.91      1.00      0.95        62
           2       0.98      0.95      0.96        56
           3       0.98      0.98      0.98        50
           4       1.00      0.99      0.99        78
           5       0.98      1.00      0.99        58
           6       1.00      1.00      1.00        63

    accuracy                           0.98       423
   macro avg       0.98      0.98      0.98       423
weighted avg       0.98      0.98      0.98       423

[[52  4  0  0  0  0  0]
 [ 0 62  0  0  0  0  0]
 [ 0  2 53  1  0  0  0]
 [ 0  0  1 49  0  0  0]
 [ 0  0  0  0 77  1  0]
 [ 0  0  0  0  0 58  0]
 [ 0  0  0  0  0  0 63]]


## Validação Cruzada

In [29]:
# importando CrossValidation
from sklearn.model_selection import cross_val_score

In [30]:
# Aplica o CrossValidation para avaliar o desempenho do modelo RandomForest e Xgboost
cv_rf = cross_val_score(model, X, y, cv=5)
cv_xgb = cross_val_score(model_xgb, X, y-1, cv=5) # o y-1 é para ajustar os rótulos para o XGBoost


g:\Meu Drive\pos tech FIAP\fase 4\tech challenge 4\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [12:33:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
g:\Meu Drive\pos tech FIAP\fase 4\tech challenge 4\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [12:33:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
g:\Meu Drive\pos tech FIAP\fase 4\tech challenge 4\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [12:33:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
g:\Meu Drive\pos tech FIAP\fase 4\tech challenge 4\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [12:33:10] WARNING: C:\a

In [31]:
# exibindo as métricas de validação cruzada
print("=== VALIDAÇÃO CRUZADA - RandomForest ===")
print("Média do CrossValidation:", cv_rf.mean())
print("Desvio padrão do CrossValidation:", cv_rf.std())

print("\n=== VALIDAÇÃO CRUZADA - XGBoost ===")
print("Média do CrossValidation:", cv_xgb.mean())
print("Desvio padrão do CrossValidation:", cv_xgb.std())

=== VALIDAÇÃO CRUZADA - RandomForest ===
Média do CrossValidation: 0.954116948449912
Desvio padrão do CrossValidation: 0.07070167186635647

=== VALIDAÇÃO CRUZADA - XGBoost ===
Média do CrossValidation: 0.9777475266937806
Desvio padrão do CrossValidation: 0.01536449562267421


Xgboost teve maior media de acerto e menor desvio padrão

# Salvando os dados

In [32]:
# baixando o dataframe normalizado
df_nomalizado_2.to_csv("df_nomalized.csv", index=False)
df_dicionario.to_csv("df_dicionario_normalized.csv", index=False)

In [33]:
# importando o joblib para salvar os modelos
import joblib   

In [34]:
# baixa o modelo XGBoost
joblib.dump(model_xgb, "modelo_xgboost.pkl")

['modelo_xgboost.pkl']

In [37]:
# exportando o standard scaler usado para normalizar os dados
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

# Gerando arquivo com requirements

In [35]:
# exibindo a versao de todas as bibliotecas usadas
# !pip list

In [36]:
# gerando os requisitos do projeto
# !pip freeze > requirements.txt